Needed a sampling because of gpu usage quota.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/processed_dataset.csv")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48488 entries, 0 to 48487
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   prompt              48488 non-null  str  
 1   ai_target           48488 non-null  int64
 2   source_model        48488 non-null  str  
 3   text                48462 non-null  str  
 4   text30              48462 non-null  str  
 5   text30_word_count   48488 non-null  int64
 6   text50              48462 non-null  str  
 7   text50_word_count   48488 non-null  int64
 8   text100             48462 non-null  str  
 9   text100_word_count  48488 non-null  int64
 10  text200             48462 non-null  str  
 11  text200_word_count  48488 non-null  int64
 12  word_count          48488 non-null  int64
 13  word_count2         48488 non-null  int64
 14  is_match            48488 non-null  bool 
dtypes: bool(1), int64(7), str(7)
memory usage: 269.9 MB


## 1.Cleaning up columns and empty and null entries

In [3]:
df = df[df["text"].notna() & (df["text"].str.strip() != "")]

df.info()

<class 'pandas.DataFrame'>
Index: 48462 entries, 0 to 48487
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   prompt              48462 non-null  str  
 1   ai_target           48462 non-null  int64
 2   source_model        48462 non-null  str  
 3   text                48462 non-null  str  
 4   text30              48462 non-null  str  
 5   text30_word_count   48462 non-null  int64
 6   text50              48462 non-null  str  
 7   text50_word_count   48462 non-null  int64
 8   text100             48462 non-null  str  
 9   text100_word_count  48462 non-null  int64
 10  text200             48462 non-null  str  
 11  text200_word_count  48462 non-null  int64
 12  word_count          48462 non-null  int64
 13  word_count2         48462 non-null  int64
 14  is_match            48462 non-null  bool 
dtypes: bool(1), int64(7), str(7)
memory usage: 270.3 MB


In [4]:
df = df.drop(columns=["prompt", "word_count2", "is_match", "text30_word_count", "text50_word_count", "text100_word_count", "text200_word_count", "word_count", "text"])

df.info()

<class 'pandas.DataFrame'>
Index: 48462 entries, 0 to 48487
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   ai_target     48462 non-null  int64
 1   source_model  48462 non-null  str  
 2   text30        48462 non-null  str  
 3   text50        48462 non-null  str  
 4   text100       48462 non-null  str  
 5   text200       48462 non-null  str  
dtypes: int64(1), str(5)
memory usage: 114.1 MB


In [5]:
df.head()

,ai_target,source_model,text30,text50,text100,text200
0,0,Human_story,Comments The U.S. bombings thatended World War...,Comments The U.S. bombings thatended World War...,Comments The U.S. bombings thatended World War...,Comments The U.S. bombings thatended World War...
1,0,Human_story,new video loaded:Messages From Quarantine tran...,new video loaded:Messages From Quarantine tran...,new video loaded:Messages From Quarantine tran...,new video loaded:Messages From Quarantine tran...
2,0,Human_story,"Supported by Roberta Karmel, First Woman Named...","Supported by Roberta Karmel, First Woman Named...","Supported by Roberta Karmel, First Woman Named...","Supported by Roberta Karmel, First Woman Named..."
3,0,Human_story,"Supported by Contests Summer Reading Contest, ...","Supported by Contests Summer Reading Contest, ...","Supported by Contests Summer Reading Contest, ...","Supported by Contests Summer Reading Contest, ..."
4,0,Human_story,The Week on Instagram @heislerphoto was one of...,The Week on Instagram @heislerphoto was one of...,The Week on Instagram @heislerphoto was one of...,The Week on Instagram @heislerphoto was one of...


## Sampling the dataset

Shuffling the data:

In [6]:
df = df.sample(frac=1, random_state=123).reset_index(drop=True)

In [7]:
df["source_model"].value_counts()

source_model
Human_story                          7295
llama-8B                             7211
accounts/yi-01-ai/models/yi-large    7118
GPT_4-o                              7032
mistral-7B                           6942
qwen-2-72B                           6781
gemma-2-9b                           6083
Name: count, dtype: int64

In [8]:
class_probs = df["source_model"].value_counts(normalize=True)
weights = df["source_model"].map(class_probs)

weights.value_counts()

source_model
0.150530    7295
0.148797    7211
0.146878    7118
0.145103    7032
0.143246    6942
0.139924    6781
0.125521    6083
Name: count, dtype: int64

Deciding on 10% of the data.

In [9]:
n_samples = int(len(df) * 0.1)
n_samples

4846

In [10]:
sampled_df = df.sample(
    n=n_samples,
    weights=weights,
    random_state=123
)
sampled_df.info()

<class 'pandas.DataFrame'>
Index: 4846 entries, 33749 to 9248
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   ai_target     4846 non-null   int64
 1   source_model  4846 non-null   str  
 2   text30        4846 non-null   str  
 3   text50        4846 non-null   str  
 4   text100       4846 non-null   str  
 5   text200       4846 non-null   str  
dtypes: int64(1), str(5)
memory usage: 11.4 MB


In [11]:
sampled_df["source_model"].value_counts()

source_model
Human_story                          790
llama-8B                             778
accounts/yi-01-ai/models/yi-large    725
GPT_4-o                              713
qwen-2-72B                           683
mistral-7B                           647
gemma-2-9b                           510
Name: count, dtype: int64

Alternative if we want 50% human, 50% AI with balance between models.

In [12]:
target_class = "Human_story"

human_df = df[df["source_model"] == target_class]
other_df = df[df["source_model"] != target_class]

other_classes = other_df["source_model"].unique()

n_human = n_samples // 2
n_other_each = (n_samples - n_human) // len(other_classes)

samples = [
    human_df.sample(
        n=min(n_human, len(human_df)),
        random_state=123
    )
]

for cls in other_classes:
    cls_df = other_df[other_df["source_model"] == cls]
    samples.append(
        cls_df.sample(
            n=min(n_other_each, len(cls_df)),
            random_state=123
        )
    )

sampled_50_df = pd.concat(samples).sample(frac=1, random_state=123).reset_index(drop=True)

sampled_50_df["source_model"].value_counts()

source_model
Human_story                          2423
gemma-2-9b                            403
accounts/yi-01-ai/models/yi-large     403
mistral-7B                            403
GPT_4-o                               403
qwen-2-72B                            403
llama-8B                              403
Name: count, dtype: int64

## Export the datasets

First, the dataset with preserved weights

In [ ]:
# file_name = 'sampled_dataset.csv'
# sampled_df.to_csv(f"../data/{file_name}", index=False)
# print(f"DataFrame successfully exported to {file_name}")

DataFrame successfully exported to sampled_dataset.csv


Now, the one  with 1-to-1 human-ai

In [ ]:
# file_name = 'sampled_50_dataset.csv'
# sampled_50_df.to_csv(f"../data/{file_name}", index=False)
# print(f"DataFrame successfully exported to {file_name}")

DataFrame successfully exported to sampled_50_dataset.csv
